# GTDB-Tk CTS Demo

End-to-end demo of the cdm_gtdbtk CTS tool: taxonomic classification of bacterial / archaeal genomes against GTDB R232.

- **Image:** `ghcr.io/kbaseincubator/cdm_gtdbtk:0.1.1@sha256:3dd3ae33859f7719b95f69c286d7a631863c95799157824faa5dfcc6d9233673`
- **Refdata UUID:** `bb6352b4-b86f-4e3d-a858-4bc77327ab13`
- **Refdata file:** `cts-refdata/gtdbtk/r232/gtdbtk_r232_data.tar.gz` (60.8 GB compressed, ~94 GB unpacked)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/gtdbtk/test/v1`

Inputs: the same 4 test assemblies (.fna.gz) we use across the other tools. `classify_wf` operates on a genome directory (not one-file-per-container), so we use `num_containers=1` and let CTS mount all 4 inputs at a single `/input` directory that we pass to `--genome_dir`.

Memory budget: 128 GB. skani needs to load the ~75 GB `sketches.db` into memory for the ANI step.

## 1. Setup

In [1]:
import io, json, time
import pandas as pd

tscli  = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_gtdbtk:0.1.1@sha256:3dd3ae33859f7719b95f69c286d7a631863c95799157824faa5dfcc6d9233673"
OUTPUT_DIR = "cts/io/jplfaria/output/gtdbtk/test/v1"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Same 4 NCBI assemblies the other tool demos use (`cts/io/gavin/test_files/`).

In [1]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True):
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")

4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. Submit GTDB-Tk classify_wf job

Single container, all 4 inputs mounted at `/input`. classify_wf builds one shared marker-gene alignment across all inputs and places them on the same tree. Expect 1-4 hours wall time.

In [1]:
if not input_files:
    raise RuntimeError("no input genomes found")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    input_mount_point="/input",
    output_mount_point="/out",
    args=[
        "classify_wf",
        "--genome_dir", "/input",
        "--out_dir", "/out",
        "--cpus", "8",
        "--extension", "fna.gz",
    ],
    num_containers=1,
    cpus=8,
    memory="128GB",
    runtime="PT6H",
)
print(f"Job ID: {job.id}")

Job ID: a80243d6-c05e-4f07-a5a8-73e11dd620aa


## 4. Wait for completion

This is slow. classify_wf does: prodigal gene-calling on each genome, TIGRFAM + Pfam HMMER, marker alignment, pplacer placement, skani ANI vs the full GTDB representative set. Expect 1-4 hours.

In [1]:
t0 = time.time()
result = job.wait_for_completion()
print(f"completed in {(time.time()-t0)/60:.1f} min")
print(f"state: {job.get_job_status()['state']}")
print(f"exit codes: {job.get_exit_codes().get('exit_codes')}")

completed in 3.7 min
state: complete
exit codes: [0]


## 5. Inspect output files

In [2]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} total output files")

# gtdbtk writes summary TSVs in three places:
#   <root>/gtdbtk.<domain>.summary.tsv                 <- top-level final  (KEEP)
#   <root>/classify/gtdbtk.<domain>.summary.tsv        <- internal copy    (skip)
#   <root>/classify/ani_screen/...ani_summary.tsv      <- ANI intermediate (skip)
# Filter to the top-level finals only.
summaries = [
    o for o in outs
    if (o["file"].endswith("gtdbtk.bac120.summary.tsv") or o["file"].endswith("gtdbtk.ar53.summary.tsv"))
    and "/classify/" not in o["file"]
    and "/identify/" not in o["file"]
    and "/align/" not in o["file"]
]
print(f"\nclassification summary files (top-level only): {len(summaries)}")
for o in summaries:
    print(f"  {o['file']}")

9 total output files

classification summary files (top-level only): 2
  cts/io/jplfaria/output/gtdbtk/test/v1/0/gtdbtk.ar53.summary.tsv
  cts/io/jplfaria/output/gtdbtk/test/v1/0/gtdbtk.bac120.summary.tsv


## 6. Read classification summary TSVs

GTDB-Tk produces per-domain summary TSVs: `gtdbtk.bac120.summary.tsv` for bacterial classifications and `gtdbtk.ar53.summary.tsv` for archaeal. Columns include `user_genome, classification, closest_genome_reference, closest_genome_ani, closest_genome_af, msa_percent, red_value, warnings`.

In [3]:
frames = []
for o in summaries:
    bucket, key = o["file"].split("/", 1)
    raw = mincli.get_object(bucket, key).read().decode("utf-8")
    df = pd.read_csv(io.StringIO(raw), sep="\t")
    df["source_file"] = o["file"]
    frames.append(df)
all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total classified genomes: {len(all_df)}")
if len(all_df):
    print("\nClassifications (truncated):")
    for _, r in all_df.iterrows():
        print(f"  {r['user_genome']}: {r['classification']}")
all_df[[c for c in all_df.columns if c in [
    "user_genome", "classification", "closest_genome_reference",
    "closest_genome_ani", "closest_genome_af", "msa_percent"]]].head(10)

Total classified genomes: 4

Classifications (truncated):
  GCA_000008085.1_ASM808v1_genomic: d__Archaea;p__Nanobdellota;c__Nanobdellia;o__Nanobdellales;f__Nanoarchaeaceae;g__Nanoarchaeum;s__Nanoarchaeum equitans
  GCA_000145985.1_ASM14598v1_genomic: d__Archaea;p__Thermoproteota;c__Thermoprotei_A;o__Sulfolobales;f__Ignisphaeraceae;g__Ignisphaera;s__Ignisphaera aggregans
  GCA_000010565.1_ASM1056v1_genomic: d__Bacteria;p__Bacillota;c__Desulfotomaculia;o__Desulfotomaculales;f__Pelotomaculaceae;g__Pelotomaculum;s__Pelotomaculum thermopropionicum
  GCA_000147015.1_ASM14701v1_genomic: d__Bacteria;p__Pseudomonadota;c__Gammaproteobacteria;o__Burkholderiales;f__Burkholderiaceae;g__Zinderia;s__Zinderia insecticola


                          user_genome  \
0    GCA_000008085.1_ASM808v1_genomic   
1  GCA_000145985.1_ASM14598v1_genomic   
2   GCA_000010565.1_ASM1056v1_genomic   
3  GCA_000147015.1_ASM14701v1_genomic   

                                      classification closest_genome_reference  \
0  d__Archaea;p__Nanobdellota;c__Nanobdellia;o__N...          GCA_000008085.1   
1  d__Archaea;p__Thermoproteota;c__Thermoprotei_A...          GCA_000145985.1   
2  d__Bacteria;p__Bacillota;c__Desulfotomaculia;o...          GCA_000010565.1   
3  d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...          GCA_000147015.1   

   closest_genome_ani  closest_genome_af  msa_percent  
0               100.0                1.0          NaN  
1               100.0                1.0          NaN  
2               100.0                1.0          NaN  
3               100.0                1.0          NaN  

## 7. End-to-end check

In [2]:
# Drop any rows where classification is missing (empty domain files write a header-only row).
classified = all_df[all_df["classification"].notna()] if len(all_df) else all_df

checks = {
    "job complete":                          job.get_job_status()["state"] == "complete",
    "has summary TSV(s)":                    len(summaries) >= 1,
    "non-empty classifications":             len(classified) > 0,
    "covers all input genomes":              len(classified) == len(input_files),
    "every classified row has a real taxon": classified["classification"].str.startswith("d__").all() if len(classified) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
if all(checks.values()):
    print("\nAll green.")
else:
    print("\nSomething's off, inspect outputs above.")

  [x] job complete
  [x] has summary TSV(s)
  [x] non-empty classifications
  [x] covers all input genomes
  [x] every classified row has a real taxon

All green.
